# Parrotlet-A 2.5 Pro — English Audio Transcription (Colab)

This notebook loads Eka Care's `parrotlet-a-2.5-pro` speech-LLM and transcribes an uploaded English audio recording.

**Fixes baked into this notebook:**
1. `transformers` pinned to a version compatible with the model's custom `Gemma3ForConditionalGeneration` decoder (avoids the `inputs_embeds` forwarding error).
2. A patched `transcribe_fixed()` that casts audio features to the encoder's dtype (bf16) instead of upcasting the whole model to fp32 — avoids CUDA OOM on a T4.
3. Skips the library's internal resampling bug by pre-resampling audio to 16kHz with `librosa` before calling transcribe.
4. Three loading strategies for CPU-RAM-constrained sessions (see cells 2 / 2b / 2c) — try them **in order**, only moving to the next if the previous one crashes.

**If your GPU/session disconnects mid-way:** re-run cell 0 → your chosen loading cell (2, 2b, or 2c) → cell 3 → cell 4, in order. You don't need to repeat earlier successful steps from a totally different strategy.

## 0. Disable TensorFlow backend (saves RAM before anything else loads)
`transformers` checks for a TensorFlow backend on import, which costs real RAM you don't need for a pure-PyTorch model. Run this first, before any other cell — including the pip installs.

In [ ]:
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"
print("TensorFlow backend disabled for transformers.")

## 1. Install / pin dependencies
Run once per fresh Colab VM. If prompted to restart the runtime after this, do so (Runtime → Restart runtime), then continue from cell 2.

In [ ]:
!pip install "transformers==4.52.0" --quiet
!pip install librosa soxr --quiet
!pip install -U accelerate bitsandbytes --quiet

import transformers
print("transformers version:", transformers.__version__)

## 2. Load the model (try this first)
Run this after any runtime restart / GPU reconnect. Loads in bf16 automatically — do **not** call `.float()` on this model, it will OOM on a T4.

**RAM note:** loading can briefly spike CPU RAM usage during weight materialization, which can crash a standard Colab session (~12GB RAM). The cell below checks free RAM first, then loads with `low_cpu_mem_usage=True` and explicit bf16 dtype to keep peak RAM as low as possible.

**If this cell crashes with "session crashed after using all available RAM" — skip to cell 2b.**

In [ ]:
# Check available RAM before loading (run right after a fresh runtime restart for a clean reading)
!free -h

from transformers import AutoModel
import torch, gc

gc.collect()
torch.cuda.empty_cache()

MODEL_ID = "ekacare/parrotlet-a-2.5-pro"

print("Loading Parrotlet-A...")

model = AutoModel.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)

print("=" * 70)
print("PARROTLET-A LOADED")
print("=" * 70)
print("Model class:", type(model))
print("Device:", next(model.parameters()).device)
print("Dtype:", next(model.parameters()).dtype)

## 2b. If cell 2 crashes on RAM — staged loading + 4-bit decoder quantization

Run this **instead of** cell 2 if you hit "session crashed after using all available RAM."

Two changes from the plain loader:
1. Encoder, decoder, projector are loaded **one at a time**, each moved to GPU and CPU-cleaned before the next starts — so peak RAM is only ever the size of the single largest component, not all three combined.
2. The decoder (the ~4B-parameter Gemma3/MedGemma component — by far the largest piece) is loaded in **4-bit** via `bitsandbytes`, cutting its memory footprint roughly 4x.

**Trade-off:** 4-bit quantization can very slightly reduce transcription quality/coherence compared to full bf16, though for ASR-style generation this is usually minor.

**If this still crashes on RAM — skip to cell 2c.**

In [ ]:
!free -h

import os, glob, json, gc, torch, sys
from huggingface_hub import snapshot_download
from safetensors.torch import load_file
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer, AutoProcessor, AutoConfig, BitsAndBytesConfig

MODEL_ID = "ekacare/parrotlet-a-2.5-pro"
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Downloading repo snapshot...")
local_dir = snapshot_download(repo_id=MODEL_ID)

encoder_dir = os.path.join(local_dir, "encoder")
decoder_dir = os.path.join(local_dir, "decoder")
projector_dir = os.path.join(local_dir, "projector")

# ---- 1. Encoder ----
print("Loading encoder...")
encoder = AutoModel.from_pretrained(
    encoder_dir, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True
).encoder.to(device)

safetensors_files = glob.glob(f"{encoder_dir}/*.safetensors")
combined_state_dict = {}
for fp in sorted(safetensors_files):
    combined_state_dict.update(load_file(fp))
encoder.load_state_dict(combined_state_dict)
encoder.eval()
del combined_state_dict
gc.collect(); torch.cuda.empty_cache()
print("Encoder loaded ->", device)

# ---- 2. Decoder (4-bit) ----
print("Loading decoder in 4-bit (largest component)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

decoder = AutoModelForCausalLM.from_pretrained(
    decoder_dir,
    quantization_config=bnb_config,
    low_cpu_mem_usage=True,
    device_map={"": 0},  # place directly on GPU 0, skips full CPU staging
)
decoder.eval()
gc.collect(); torch.cuda.empty_cache()
print("Decoder loaded ->", device)

tokenizer = AutoTokenizer.from_pretrained(decoder_dir)
processor = AutoProcessor.from_pretrained(encoder_dir)

# ---- 3. Projector ----
print("Loading projector...")
with open(os.path.join(projector_dir, "config.json"), "r") as f:
    proj_cfg = json.load(f)

custom_module_name = [m for m in sys.modules if "modelling_speech" in m or "modeling_speech" in m]
if custom_module_name:
    speech_mod = sys.modules[custom_module_name[0]]
else:
    from transformers.dynamic_module_utils import get_class_from_dynamic_module
    SpeechLLMClass = get_class_from_dynamic_module(
        class_reference=f"{MODEL_ID}--modelling_speech-llm.SpeechLLM",
        pretrained_model_name_or_path=MODEL_ID,
    )
    speech_mod = sys.modules[SpeechLLMClass.__module__]

load_projector = speech_mod.load_projector
SpeechLLMConfig = speech_mod.SpeechLLMConfig
SpeechLLM = speech_mod.SpeechLLM

encoder_dim = proj_cfg["encoder_dim"]
llm_dim = proj_cfg["llm_dim"]
linear_hidden_dim = proj_cfg["linear_hidden_dim"]
k = proj_cfg["encoder_projector_ds_rate"]
projector = load_projector(projector_dir, encoder_dim, llm_dim, linear_hidden_dim, k).to(device)
projector.eval()
gc.collect(); torch.cuda.empty_cache()
print("Projector loaded ->", device)

# ---- 4. Assemble ----
config = AutoConfig.from_pretrained(local_dir, trust_remote_code=True)
sampling_rate = getattr(processor.feature_extractor, "sampling_rate", 16000)
model = SpeechLLM(config, encoder, projector, decoder, tokenizer, processor, sampling_rate)

print("=" * 70)
print("PARROTLET-A LOADED (staged, 4-bit decoder)")
print("=" * 70)
print("Model class:", type(model))
print("Device:", next(model.parameters()).device)

## 2c. If 2b still crashes on RAM — meta-device streaming (most memory-efficient option)

This is the most advanced fallback. Instead of materializing the decoder's weights in CPU RAM at any point, it:
1. Builds the model architecture on a **meta device** (`init_empty_weights()`) — this creates the model's structure with zero real memory, just shapes and dtypes, no actual tensor data.
2. Streams each weight tensor **directly from disk to GPU** via `load_checkpoint_and_dispatch`, so CPU RAM is never used as a staging area at all.

Combines with 4-bit quantization for maximum headroom.

**Note on `no_split_module_classes`:** this should be the exact class name of this model's repeated transformer block. I've guessed `"Gemma3DecoderLayer"` based on the Gemma3/MedGemma architecture family, but this custom checkpoint may use a different name. If this cell errors specifically on that line or gives a confusing device-placement error, the safest fix is to **delete the `no_split_module_classes=[...]` line entirely** and let `device_map="auto"` figure out placement on its own — slightly less optimal, but it will still run.

In [ ]:
!free -h

import os, glob, json, gc, torch, sys
from huggingface_hub import snapshot_download
from safetensors.torch import load_file
from transformers import (
    AutoModel, AutoModelForCausalLM, AutoTokenizer, AutoProcessor,
    AutoConfig, BitsAndBytesConfig,
)
from accelerate import init_empty_weights, load_checkpoint_and_dispatch

MODEL_ID = "ekacare/parrotlet-a-2.5-pro"
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Downloading repo snapshot...")
local_dir = snapshot_download(repo_id=MODEL_ID)

encoder_dir = os.path.join(local_dir, "encoder")
decoder_dir = os.path.join(local_dir, "decoder")
projector_dir = os.path.join(local_dir, "projector")

# ---- 1. Encoder (small enough to load normally) ----
print("Loading encoder...")
encoder = AutoModel.from_pretrained(
    encoder_dir, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True
).encoder.to(device)

safetensors_files = glob.glob(f"{encoder_dir}/*.safetensors")
combined_state_dict = {}
for fp in sorted(safetensors_files):
    combined_state_dict.update(load_file(fp))
encoder.load_state_dict(combined_state_dict)
encoder.eval()
del combined_state_dict
gc.collect(); torch.cuda.empty_cache()
print("Encoder loaded ->", device)

# ---- 2. Decoder: meta-device streaming + 4-bit ----
print("Loading decoder via meta-device streaming (largest component)...")

decoder_config = AutoConfig.from_pretrained(decoder_dir)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

with init_empty_weights():
    decoder = AutoModelForCausalLM.from_config(decoder_config, torch_dtype=torch.bfloat16)

decoder = load_checkpoint_and_dispatch(
    decoder,
    checkpoint=decoder_dir,
    device_map="auto",
    dtype=torch.bfloat16,
    no_split_module_classes=["Gemma3DecoderLayer"],  # <-- if this errors, delete this line and retry
)
decoder.eval()
gc.collect(); torch.cuda.empty_cache()
print("Decoder loaded ->", device)

tokenizer = AutoTokenizer.from_pretrained(decoder_dir)
processor = AutoProcessor.from_pretrained(encoder_dir)

# ---- 3. Projector ----
print("Loading projector...")
with open(os.path.join(projector_dir, "config.json"), "r") as f:
    proj_cfg = json.load(f)

custom_module_name = [m for m in sys.modules if "modelling_speech" in m or "modeling_speech" in m]
if custom_module_name:
    speech_mod = sys.modules[custom_module_name[0]]
else:
    from transformers.dynamic_module_utils import get_class_from_dynamic_module
    SpeechLLMClass = get_class_from_dynamic_module(
        class_reference=f"{MODEL_ID}--modelling_speech-llm.SpeechLLM",
        pretrained_model_name_or_path=MODEL_ID,
    )
    speech_mod = sys.modules[SpeechLLMClass.__module__]

load_projector = speech_mod.load_projector
SpeechLLMConfig = speech_mod.SpeechLLMConfig
SpeechLLM = speech_mod.SpeechLLM

encoder_dim = proj_cfg["encoder_dim"]
llm_dim = proj_cfg["llm_dim"]
linear_hidden_dim = proj_cfg["linear_hidden_dim"]
k = proj_cfg["encoder_projector_ds_rate"]
projector = load_projector(projector_dir, encoder_dim, llm_dim, linear_hidden_dim, k).to(device)
projector.eval()
gc.collect(); torch.cuda.empty_cache()
print("Projector loaded ->", device)

# ---- 4. Assemble ----
config = AutoConfig.from_pretrained(local_dir, trust_remote_code=True)
sampling_rate = getattr(processor.feature_extractor, "sampling_rate", 16000)
model = SpeechLLM(config, encoder, projector, decoder, tokenizer, processor, sampling_rate)

print("=" * 70)
print("PARROTLET-A LOADED (meta-device streaming + 4-bit decoder)")
print("=" * 70)
print("Model class:", type(model))
print("Decoder device map:", getattr(model.decoder, "hf_device_map", "n/a"))

## 3. Patched transcription function
Same logic as the model's own `transcribe()`, with one fix: casts the audio features to the encoder's dtype (bf16) before the forward pass, instead of upcasting the whole model to fp32.

In [ ]:
import torch

def transcribe_fixed(model, audio, orig_sr, max_new_tokens=256, repetition_penalty=1.2, **gen_kwargs):
    device = next(model.parameters()).device
    model_dtype = next(model.encoder.parameters()).dtype  # bf16

    prompt = model.get_prompt()
    input_ids = torch.tensor(model.tokenizer(prompt, add_special_tokens=False)['input_ids'])
    input_attention_mask = torch.ones_like(input_ids)

    audio_token = model.tokenizer.convert_tokens_to_ids(model.audio_token)
    audio_pos = input_ids.tolist().index(audio_token)

    input_ids = input_ids.unsqueeze(0).to(device)
    input_attention_mask = input_attention_mask.unsqueeze(0).to(device)

    processed_audio = model.preprocess_audio(audio, orig_sr)

    audio_features = model.processor.feature_extractor(
        [processed_audio], sampling_rate=model.sampling_rate, return_tensors="pt"
    ).input_features
    audio_features = audio_features.to(device=device, dtype=model_dtype)  # the fix

    with torch.no_grad():
        audio_embeddings = model.encoder(audio_features).last_hidden_state
        projected_audio_embeddings = model.projector(audio_embeddings)

    input_embeddings = model.decoder.get_input_embeddings()(input_ids)
    batch_size, input_seq_len, embed_dim = input_embeddings.shape
    audio_seq_len = projected_audio_embeddings.shape[1]

    max_combined_len = input_seq_len + audio_seq_len - 1
    combined_embeddings = torch.zeros(batch_size, max_combined_len, embed_dim, device=device, dtype=input_embeddings.dtype)
    combined_attention_mask = torch.zeros(batch_size, max_combined_len, device=device, dtype=input_attention_mask.dtype)

    combined_embeddings[:, :audio_pos] = input_embeddings[:, :audio_pos]
    combined_attention_mask[:, :audio_pos] = input_attention_mask[:, :audio_pos]
    combined_embeddings[:, audio_pos:audio_pos+audio_seq_len] = projected_audio_embeddings
    combined_attention_mask[:, audio_pos:audio_pos+audio_seq_len] = 1

    suffix_start = audio_pos + 1
    suffix_len = input_seq_len - suffix_start
    out_start = audio_pos + audio_seq_len
    combined_embeddings[:, out_start:out_start+suffix_len] = input_embeddings[:, suffix_start:]
    combined_attention_mask[:, out_start:out_start+suffix_len] = input_attention_mask[:, suffix_start:]

    default_gen_kwargs = {
        'max_new_tokens': max_new_tokens,
        'do_sample': False,
        'repetition_penalty': repetition_penalty,
        'pad_token_id': model.tokenizer.pad_token_id,
        'eos_token_id': model.tokenizer.eos_token_id,
    }
    default_gen_kwargs.update(gen_kwargs)

    with torch.no_grad():
        outputs = model.decoder.generate(
            inputs_embeds=combined_embeddings,
            attention_mask=combined_attention_mask,
            **default_gen_kwargs
        )

    return model.tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

print("transcribe_fixed() ready.")

## 4. Upload audio and transcribe
Upload an English recording (wav/mp3/etc). It's resampled to the model's required 16kHz before transcription, sidestepping the library's internal resampling bug.

In [ ]:
import librosa
from google.colab import files

uploaded = files.upload()
file_path = list(uploaded.keys())[0]

target_sr = model.sampling_rate  # 16000
audio_array, sr = librosa.load(file_path, sr=target_sr)

print(f"Loaded '{file_path}' — {len(audio_array)/sr:.1f}s at {sr}Hz")

transcript = transcribe_fixed(model, audio_array, sr)

print("=" * 70)
print("TRANSCRIPT")
print("=" * 70)
print(transcript)

## 5. (Optional) Save transcript to a text file

In [ ]:
out_name = file_path.rsplit(".", 1)[0] + "_transcript.txt"
with open(out_name, "w", encoding="utf-8") as f:
    f.write(transcript)

print(f"Saved to {out_name}")

from google.colab import files as colab_files
colab_files.download(out_name)